# Arnold's Nucleoformer

You will definitely not be able to run this on your computer without downloading a bunch of stuff. Please see the cell output below for evidence that the program is working.

In [1]:
from aligner import GeneLoader
from genetic_algorithm import NSGA_II
from HMM import HMM
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoConfig
import numpy as np
from tqdm import tqdm
import polars as pl
import umap
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
LOADER = GeneLoader("hg38_primary")

shape: (6, 8)
┌───────┬───────┬───────────┬───────────┬───────┬──────────────┬─────────────┬───────────┐
│ index ┆ state ┆ start     ┆ end       ┆ len   ┆ chrom        ┆ chrom_start ┆ chrom_end │
│ ---   ┆ ---   ┆ ---       ┆ ---       ┆ ---   ┆ ---          ┆ ---         ┆ ---       │
│ u32   ┆ u8    ┆ i64       ┆ i64       ┆ i64   ┆ str          ┆ i64         ┆ i64       │
╞═══════╪═══════╪═══════════╪═══════════╪═══════╪══════════════╪═════════════╪═══════════╡
│ 80151 ┆ 9     ┆ 248930191 ┆ 248936570 ┆ 6380  ┆ NC_000001.11 ┆ 248930191   ┆ 248936570 │
│ 80152 ┆ 4     ┆ 248936571 ┆ 248937098 ┆ 528   ┆ NC_000001.11 ┆ 248936571   ┆ 248937098 │
│ 80153 ┆ 9     ┆ 248937099 ┆ 248956421 ┆ 19323 ┆ NC_000001.11 ┆ 248937099   ┆ 248956421 │
│ 80154 ┆ 9     ┆ 248956423 ┆ 248995235 ┆ 38813 ┆ NC_000002.12 ┆ 1           ┆ 38813     │
│ 80155 ┆ 6     ┆ 248995236 ┆ 248998048 ┆ 2813  ┆ NC_000002.12 ┆ 38814       ┆ 41626     │
│ 80156 ┆ 8     ┆ 248998049 ┆ 248998539 ┆ 491   ┆ NC_000002.12 ┆ 41627      

In [3]:
HMM_MODEL = HMM(n_states=len(LOADER.states), k=6, n_bases=6, model_name="supervised_10k_oldschema", fresh_start=True)

In [4]:
model_name = "zhihan1996/DNABERT-2-117M"

# 1. Manually set the config to disable EVERY hardware accelerator
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
config.use_triton = False
config.use_flash_attn = False
config.attn_implementation = "eager" # The most basic, stable CPU path

# 2. Load the model with this specific config
dna_bert_2_model = AutoModel.from_pretrained(
    model_name,
    config=config,
    trust_remote_code=True,
    device_map={"": "cpu"},
    torch_dtype=torch.float32 
)

dna_bert_2_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Model recovered and running on CPU.")


`torch_dtype` is deprecated! Use `dtype` instead!
/home/ruminator/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT_hyphen_2_hyphen_117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model recovered and running on CPU.


In [5]:
objectives = {
    "cos_dist": {"reverse": False},
    "ham_dist": {"reverse": True},
    "hmm_prob": {"reverse": True}
}
GA = NSGA_II("test5", objectives, 
             hmm=HMM_MODEL, 
             loader=LOADER, 
             pop_cap=100, 
             idx=21,
             model=dna_bert_2_model, 
             tokenizer=dna_bert_2_tokenizer
             )

CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.02
ITERATIONS = range(1)

In [6]:
GA.popn.clear()
GA.load_popn()
print("init", len(GA.popn))
for i in range(len(GA.popn)):
    GA.random_mutate_entry(i)
print("init", len(GA.popn))
for i in tqdm(ITERATIONS):
    print("it", len(GA.popn))
    GA.cross_mutate_random(CROSSOVER_RATE, MUTATION_RATE)
    print("it", len(GA.popn))
    GA.update_entries()
    print("it", len(GA.popn))
    GA.truncate_popn()
    print("it", len(GA.popn))
# GA.save_popn(write=True)

init 100
init 100


  0%|          | 0/1 [00:00<?, ?it/s]

it 100
it 180
it 180


100%|██████████| 1/1 [00:17<00:00, 17.80s/it]

it 100


In [13]:
df = pl.read_parquet("popns/test.parquet")
print(df.head())
embeddings = [np.array(embedding) for embedding in df["embedding"]]

umapper = umap.UMAP(n_components=2)
embeddings_2d = umapper.fit_transform(embeddings)

fig = go.Figure(go.Scatter(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    mode="markers",
    marker=dict(size=5, opacity=0.7),
))

fig.update_layout(title="UMAP Embeddings", xaxis_title="UMAP 1", yaxis_title="UMAP 2")
fig.show()


shape: (5, 10)
┌─────────────┬────────────┬────────────┬────────────┬───┬──────┬────────┬────────────┬────────────┐
│ seq         ┆ states     ┆ conv_score ┆ embedding  ┆ … ┆ live ┆ update ┆ fit_ham_di ┆ fit_hmm_pr │
│ ---         ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---  ┆ ---    ┆ st         ┆ ob         │
│ list[str]   ┆ list[u8]   ┆ f64        ┆ list[f32]  ┆   ┆ bool ┆ bool   ┆ ---        ┆ ---        │
│             ┆            ┆            ┆            ┆   ┆      ┆        ┆ f64        ┆ f64        │
╞═════════════╪════════════╪════════════╪════════════╪═══╪══════╪════════╪════════════╪════════════╡
│ ["T", "C",  ┆ [6, 6, …   ┆ -0.275116  ┆ [-0.034277 ┆ … ┆ true ┆ false  ┆ 0.066667   ┆ 3.071992   │
│ … "C"]      ┆ 6]         ┆            ┆ ,          ┆   ┆      ┆        ┆            ┆            │
│             ┆            ┆            ┆ -0.032897, ┆   ┆      ┆        ┆            ┆            │
│             ┆            ┆            ┆ … 0.101…   ┆   ┆      ┆        ┆  

In [ ]:
reference_model = NSGA_II(None, objectives, HMM_Model, LOADER, dna_bert_2_model, dna_bert_2_tokenizer, 0, 21)
reference = reference_model.master["embedding"]

embeddings = np.array(df["embedding"].to_list())
mean_embedding = embeddings.mean(axis=0)
mean_emb_diff = np.linalg.norm(embeddings - mean_embedding, axis=1)

masked_conv = df.select([
    pl.struct(["states", "mut_array"]).map_elements(
        lambda x: float(np.mean(np.array(x["states"])[np.array(x["mut_array"], dtype=bool)])),
        return_dtype=pl.Float64
    ).alias("masked_conv_score")
]).to_series().to_numpy()

fig = go.Figure(go.Scatter(
    x=mean_emb_diff,
    y=masked_conv,
    mode="markers",
    marker=dict(size=5, opacity=0.7),
))

fig.update_layout(
    title="Mean Embedding Difference vs Conservation Score",
    xaxis_title="L2 Distance from Mean Embedding",
    yaxis_title="Mean Conservation Score (mutated positions)",
)
fig.show()

NameError: name 'LOADER' is not defined

In [16]:
fig = go.Figure(go.Scatter(
    x=df["fit_ham_dist"].to_numpy(),
    y=df["fit_hmm_prob"].to_numpy(),
    mode="markers",
    marker=dict(size=5, opacity=0.7),
))

fig.update_layout(
    title="Fit Hamming Distance vs Fit HMM Probability",
    xaxis_title="Fit Hamming Distance",
    yaxis_title="Fit HMM Probability",
)
fig.show()